In [1]:
import numpy 
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from omegaconf import OmegaConf
import json
import os
__DIR__ = os.path.dirname("../")
import sys
sys.path.append(__DIR__)
from datasets import load_from_disk, concatenate_datasets
import numpy as np
import jsonlines

from tqdm import tqdm

In [2]:
from diff_masking.utils.phi3 import create_masked_phi

In [3]:
import torch

# Set the CUDA device
torch.cuda.set_device(1)  # Replace 1 with the desired CUDA device index

In [4]:
model_name = "microsoft/Phi-3-mini-128k-instruct"
model = AutoModelForCausalLM.from_pretrained( 
            model_name,  
            device_map="cuda:1",  
            torch_dtype=torch.bfloat16,  
            trust_remote_code=True,  
            attn_implementation="flash_attention_2"
) 

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [6]:
checkpoint = "2024-10-28_17-00-51"
checkpoint_path = os.path.join("../outputs/main/", checkpoint)

In [9]:
with open(checkpoint_path+"/config.json", 'r') as file:
    config_dict = json.load(file)

In [10]:
config = OmegaConf.create(config_dict)

In [11]:
masking = config.masking if "masking" in config else "probabilistic"

In [13]:

model = create_masked_phi(model, config.specs.target_layers, config.specs.init_prob, config.specs.tau, os.path.join(checkpoint_path,"model.pth"), masking=masking)

In [14]:
model.cuda()
torch.cuda.empty_cache()

In [15]:
for l in config.specs.target_layers:
    model.model.layers._modules[str(l)].self_attn.mask_enabled = True
    model.model.layers._modules[str(l)].mlp.mask_enabled = True

In [16]:
import json

with open('questions/astronomy.json', 'r') as file:
    astronomy_questions = json.load(file)
with open('questions/biology.json', 'r') as file:
    biology_questions = json.load(file)
with open('questions/quantum.json', 'r') as file:
    quantum_questions = json.load(file)
with open('questions/rl.json', 'r') as file:
    rl_questions = json.load(file)
with open('questions/family.json', 'r') as file:
    family_questions = json.load(file)

In [17]:
generation_args = { 
        "max_new_tokens": 1024, 
        "temperature": 0.2, 
        "do_sample": True,
} 

In [20]:
os.mkdir(os.path.join("outputs", checkpoint))

In [21]:
responses = []
for k, v in tqdm(biology_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])
output_file = os.path.join("outputs", checkpoint,"biology.jsonl")

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 5/5 [03:45<00:00, 45.12s/it]


In [22]:
responses = []
for k, v in tqdm(quantum_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])
output_file = os.path.join("outputs", checkpoint,"quantum.jsonl")

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 5/5 [05:35<00:00, 67.14s/it]


In [23]:
responses = []
for k, v in tqdm(astronomy_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])

output_file = os.path.join("outputs", checkpoint,"astronomy.jsonl")

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 6/6 [10:44<00:00, 107.41s/it]


In [24]:
responses = []
for k, v in tqdm(rl_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])

output_file = os.path.join("outputs", checkpoint,"rl.jsonl")

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 5/5 [06:03<00:00, 72.73s/it] 


In [25]:
responses = []
for k, v in tqdm(family_questions.items()):
    messages = [[{"role": "user", "content": q["question"]}] for q in v]
    tokenied = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True, padding=True, return_dict=True)
    input_ids = tokenied["input_ids"]
    attn_mask = tokenied["attention_mask"]

    input_ids = input_ids.to(model.device)
    attn_mask = attn_mask.to(model.device)
        
    outputs = model.generate(input_ids=input_ids,attention_mask=attn_mask, **generation_args)
    decoded_texts = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)
    responses.extend([{"user": v[i]["question"], "assistant": decoded_texts[i]} for i in range(len(v))])

output_file = os.path.join("outputs", checkpoint,"family.jsonl")

# Open the file in write mode
with jsonlines.open(output_file, mode='w') as writer:
    # Iterate over the list and write each element as a separate line
    for item in responses:
        writer.write(item)

100%|██████████| 7/7 [05:48<00:00, 49.78s/it]
